# Review local recordings

List your own audio files, then listen to each one and annotate it.

**You need:** audio files in `local_data/audio/` (subfolders are fine), or another folder you set below. No `.env` is needed.

**Run it** in JupyterLab, from top to bottom. The review tool in step 2 doesn't work in VS Code.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
from jupyter_bioacoustic import BioacousticAnnotator

# Work from the repo folder, so every path here and in annotator_config/ starts there.
if Path.cwd().name == "notebooks":
    os.chdir("..")

## 1. List recordings

Set where your audio is and run the next two cells. They list every audio file in that folder in `local_data/metadata.csv`. Skip them to reuse the list from last time.

In [ ]:
AUDIO_DIR = "local_data/audio"  # on Windows use /, e.g. "D:/recordings"
CHUNK_SECONDS = None  # e.g. 10 splits each recording into 10 s clips (the last may be shorter); None = whole recordings

In [ ]:
AUDIO_EXTENSIONS = {".wav", ".flac", ".mp3", ".ogg"}
METADATA_CSV = Path("local_data/metadata.csv")  # the review tool in step 2 loads this file

rows = []
for path in sorted(Path(AUDIO_DIR).rglob("*")):
    if path.suffix.lower() not in AUDIO_EXTENSIONS or path.name.startswith("."):
        continue  # not audio, or a hidden macOS "._" copy
    try:
        duration = round(sf.info(path).duration, 3)
    except sf.LibsndfileError:
        duration = 0
    if duration == 0:
        print(f"Skipped unreadable or empty file: {path}")
        continue
    # The review tool plays start_time to end_time (seconds into the file), so a whole recording is 0 to its duration.
    step = CHUNK_SECONDS or duration
    for start in np.arange(0, duration, step):
        rows.append({"path": path.as_posix(), "start_time": round(start, 3), "end_time": round(min(start + step, duration), 3)})
if not rows:
    raise FileNotFoundError(f"No audio files found in {Path(AUDIO_DIR).resolve()}")

metadata = pd.DataFrame(rows)
metadata.insert(0, "id", range(1, len(metadata) + 1))
METADATA_CSV.parent.mkdir(exist_ok=True)
metadata.to_csv(METADATA_CSV, index=False)
print(f"{metadata.path.nunique()} recordings -> {len(metadata)} clips in {METADATA_CSV}")
metadata.head()

## 2. Review

Annotations are saved to `output/local_reviews.csv` as you go.

The form, spectrogram views and columns shown are set in `annotator_config/`.

In [ ]:
ann = BioacousticAnnotator(project="annotator_config/projects/local_data.yaml")
ann.open()